### Y ahora - Semana 3 Día 3

## AutoGen Core

Algo un poco diferente.

Esto es agnóstico al marco de agente subyacente

Puedes usar AutoGen AgentChat, o puedes usar algo más; es un marco de interacción de agentes.

Desde ese punto de vista, está posicionado de manera similar a LangGraph.

### El principio fundamental

Autogen Core desacopla la lógica de un agente de cómo se entregan los mensajes.  
El marco proporciona una infraestructura de comunicación, junto con el ciclo de vida del agente, y los agentes son responsables de su propio trabajo.

La infraestructura de comunicación se llama Runtime.

Hay 2 tipos: **Standalone** y **Distributed**.

Hoy usaremos un runtime standalone: el **SingleThreadedAgentRuntime**, una implementación de runtime de agente embebido local.

Mañana veremos brevemente un runtime Distributed.

In [1]:
# Importar componentes necesarios para AutoGen Core
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)

True

### Primero definimos nuestro objeto Message

Cualquier estructura que queramos para mensajes en nuestro marco de agentes.

In [2]:
# Definir una clase Message simple
@dataclass
class Message:
    content: str

### Ahora definimos nuestro Agente

Una subclase de RoutedAgent.

Cada Agente tiene un **ID de Agente** que tiene 2 componentes:  
`agent.id.type` describe el tipo de agente  
`agent.id.key` le da su identificador único

Cualquier método con el decorador `@message_handler` tendrá la oportunidad de recibir mensajes.

In [3]:
# Definir un agente simple que responde a mensajes
class SimpleAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("Simple")

    @message_handler
    async def on_my_message(self, message: Message, ctx: MessageContext) -> Message:
        return Message(content=f"Esto es {self.id.type}-{self.id.key}. Dijiste '{message.content}' y no estoy de acuerdo.")

### OK creemos un runtime Standalone y registremos nuestro tipo de agente

In [4]:
# Crear runtime y registrar agente simple
runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())

AgentType(type='simple_agent')

### ¡Genial! Vamos a iniciar un runtime y enviar un mensaje

In [5]:
# Iniciar el runtime
runtime.start()

In [6]:
# Enviar mensaje al agente y obtener respuesta
agent_id = AgentId("simple_agent", "default")
response = await runtime.send_message(Message("¡Hola!"), agent_id)
print(">>>", response.content)

>>> Esto es simple_agent-default. Dijiste '¡Hola!' y no estoy de acuerdo.


In [7]:
# Detener y cerrar el runtime
await runtime.stop()
await runtime.close()

### OK Ahora hagamos algo más interesante

Usaremos un AgenteChat Assistant!

In [8]:
# Definir un agente que usa LLM
class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent("LLMAgent", model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received message: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type} responded: {reply}")
        return Message(content=reply)

In [9]:
# Crear nuevo runtime y registrar agentes
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())
await MyLLMAgent.register(runtime, "LLMAgent", lambda: MyLLMAgent())

AgentType(type='LLMAgent')

In [10]:
# Iniciar runtime y enviar mensajes entre agentes
runtime.start()  # Start processing messages in the background.
response = await runtime.send_message(Message("Hola amigo!"), AgentId("LLMAgent", "default"))
print(">>>", response.content)
response =  await runtime.send_message(Message(response.content), AgentId("simple_agent", "default"))
print(">>>", response.content)
response = await runtime.send_message(Message(response.content), AgentId("LLMAgent", "default"))

LLMAgent received message: Hola amigo!
LLMAgent responded: ¡Hola! ¿En qué puedo ayudarte hoy?
>>> ¡Hola! ¿En qué puedo ayudarte hoy?
>>> Esto es simple_agent-default. Dijiste '¡Hola! ¿En qué puedo ayudarte hoy?' y no estoy de acuerdo.
LLMAgent received message: Esto es simple_agent-default. Dijiste '¡Hola! ¿En qué puedo ayudarte hoy?' y no estoy de acuerdo.
LLMAgent responded: Entiendo, ¿hay algo específico sobre lo que te gustaría hablar o discutir? Estoy aquí para ayudar.


In [11]:
# Detener y cerrar el runtime
await runtime.stop()
await runtime.close()

### OK ahora mostremos esto en funcionamiento - ¡hagamos que 3 agentes interactúen!

In [12]:
# Importar Ollama para agentes con diferentes modelos
from autogen_ext.models.ollama import OllamaChatCompletionClient

# Definir agentes para piedra, papel o tijera
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [17]:
# Definir juez para el juego
JUDGE = "Eres el árbitro de un juego de piedra, papel o tijera. Los jugadores han hecho estas elecciones:\n"

class RockPaperScissorsAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        instruction = "Estás jugando a piedra, papel o tijera. Responde solo con una de estas palabras: piedra, papel o tijera."
        message = Message(content=instruction)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)
        response2 = await self.send_message(message, inner_2)
        result = f"Player 1: {response1.content}\nPlayer 2: {response2.content}\n"
        judgement = f"{JUDGE}{result}¿Quién gana?"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + response.chat_message.content)

In [18]:
# Registrar agentes en el runtime
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RockPaperScissorsAgent.register(runtime, "rock_paper_scissors", lambda: RockPaperScissorsAgent("rock_paper_scissors"))
runtime.start()

In [19]:
# Ejecutar el juego y mostrar resultado
agent_id = AgentId("rock_paper_scissors", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

Player 1: papel
Player 2: piedra
El jugador 1 gana, ya que el papel envuelve a la piedra. TERMINATE.


In [16]:
# Detener y cerrar el runtime
await runtime.stop()
await runtime.close()